# Fase 3 — Regras de Associação: Candidatos 2026 (TSE)

## Pergunta de negócio

> **Que combinações de características dos candidatos a Deputado Federal na Região Norte aparecem juntas com mais frequência do que se fossem independentes? Em particular: quais atributos (partido, gênero, escolaridade, cor/raça, UF de nascimento, UF de candidatura) estão associados a cada perfil identificado na Fase 2 — e, assim que a eleição ocorrer, a ser eleito?**

Até aqui, a clusterização (Fase 2) agrupou candidatos por **distância** em variáveis numéricas (idade, escolaridade, patrimônio). Regras de associação fazem uma pergunta diferente: em vez de "quem está perto de quem", **quais combinações de atributos categóricos aparecem juntas com mais frequência do que o acaso explicaria**? A técnica clássica pra isso é o **Apriori**, que originalmente resolve o problema da "cesta de compras" (que produtos são comprados juntos) — aqui, cada candidato é uma cesta, e cada valor de cada variável categórica (partido, gênero, escolaridade...) é um item dessa cesta.

## Configuração

Use os **mesmos valores** que você usou nas Fases 0/1/2, para carregar o arquivo certo.


In [13]:
CARGO = "DEPUTADO FEDERAL"   # mesmo valor usado nas Fases 0/1/2
UF = ["AC", "AP", "AM", "PA", "RO", "RR", "TO"]   # mesmo valor usado nas Fases 0/1/2 (lista = Região Norte)
NOME_REGIAO = "REGIAO_NORTE"   # mesmo valor usado na Fase 0 (só usado no nome do arquivo quando UF é uma lista/região)
ANO_ELEICAO = 2026

## Preparando a base — reconstituindo os clusters nomeados da Fase 2

Apriori precisa de **itens categóricos**, não de distância — mas queremos incluir o cluster da Fase 2 como mais um item da cesta (pra achar regras do tipo "esse perfil de candidato é dos Com patrimônio"). Recalculamos aqui, rapidamente, a mesma partição usada na Fase 2 (`02_clusterizacao.ipynb`) — mesma lógica de auto-suficiência do notebook original: **Bisecting K-Means com k=3** (critério de inércia média), sobre as variáveis-driver `IDADE`, `ANOS_ESTUDO` e `Total_Bens_Log`, com a mesma `SEMENTE`.

Para a Região Norte, essa partição separa os candidatos a Deputado Federal em três grupos bem diferentes entre si:

- **Com patrimônio** — 67% dos candidatos (622), os únicos com algum patrimônio declarado (mediana de R$ 451 mil).
- **Jovens sem bens** — 17% (158), o grupo mais novo (41 anos em média) e de menor escolaridade média, quase sem patrimônio declarado (99% zerado) e majoritariamente solteiro.
- **Sem bens e alta escolaridade** — 16% (145), 98% sem patrimônio, a maior escolaridade média dos três — e o único dos três grupos com maioria de mulheres (52%, contra minoria masculina nos outros dois).

In [14]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from mlxtend.frequent_patterns import apriori, association_rules
from pyvis.network import Network
from IPython.display import IFrame

sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)  # não truncar antecedents/consequents nas tabelas

SEMENTE = 42
np.random.seed(SEMENTE)  # mesmo racional de reprodutibilidade defensiva dos notebooks anteriores

# mesma lógica de nomeação de arquivo usada nas Fases 0 e 2 — UF pode ser None (Brasil),
# uma sigla única, ou uma lista de siglas (região, como aqui)
if UF is None:
    nome_uf = 'BRASIL'
elif isinstance(UF, str):
    nome_uf = UF
else:
    nome_uf = NOME_REGIAO
nome_cargo = CARGO.replace(' ', '_')
arquivo = f"dados/candidatos_{nome_cargo}_{nome_uf}_{ANO_ELEICAO}.csv"

df = pd.read_csv(arquivo)
print(f"Carregado: {arquivo}  ->  {df.shape[0]} candidatos, {df.shape[1]} colunas")
df.head()

Carregado: dados/candidatos_DEPUTADO_FEDERAL_REGIAO_NORTE_2026.csv  ->  925 candidatos, 59 colunas


,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,TP_ABRANGENCIA,SG_UF,SG_UE,NM_UE,CD_CARGO,DS_CARGO,SQ_CANDIDATO,NR_CANDIDATO,NM_CANDIDATO,NM_URNA_CANDIDATO,NM_SOCIAL_CANDIDATO,NR_CPF_CANDIDATO,DS_EMAIL,CD_SITUACAO_CANDIDATURA,DS_SITUACAO_CANDIDATURA,TP_AGREMIACAO,NR_PARTIDO,SG_PARTIDO,NM_PARTIDO,NR_FEDERACAO,NM_FEDERACAO,SG_FEDERACAO,DS_COMPOSICAO_FEDERACAO,SQ_COLIGACAO,NM_COLIGACAO,DS_COMPOSICAO_COLIGACAO,SG_UF_NASCIMENTO,DT_NASCIMENTO,NR_TITULO_ELEITORAL_CANDIDATO,CD_GENERO,DS_GENERO,CD_GRAU_INSTRUCAO,DS_GRAU_INSTRUCAO,CD_ESTADO_CIVIL,DS_ESTADO_CIVIL,CD_COR_RACA,DS_COR_RACA,CD_OCUPACAO,DS_OCUPACAO,CD_SIT_TOT_TURNO,DS_SIT_TOT_TURNO,BENS IMÓVEIS,BENS MÓVEIS,DINHEIRO,INVESTIMENTOS RENDA FIXA,INVESTIMENTOS RENDA VARIÁVEL,OUTROS BENS,Total_Bens,Total_Bens_Log,IDADE
0,23/08/2026,19:30:42,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,AC,AC,ACRE,6,DEPUTADO FEDERAL,10002532416,1567,ADGICLEIA DOS SANTOS GUIMARÃES,DR.. KEL GUIMARÃES,#NULO,1138048224,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,15,MDB,MOVIMENTO DEMOCRÁTICO BRASILEIRO,-1,#NULO,#NULO,#NULO,10001800109,PARTIDO ISOLADO,MDB,AC,1991-07-22,5979012437,4,FEMININO,8,SUPERIOR COMPLETO,3,CASADO(A),3,PARDA,111,MÉDICO,-1,#NULO,0.0,60000.0,0.0,0.0,0.0,0.00,60000.00,11.002117,35
1,23/08/2026,19:30:42,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,AC,AC,ACRE,6,DEPUTADO FEDERAL,10002532417,1511,FÁBIO DE ARAÚJO FREITAS,FÁBIO ARAÚJO,#NULO,52152901215,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,15,MDB,MOVIMENTO DEMOCRÁTICO BRASILEIRO,-1,#NULO,#NULO,#NULO,10001800109,PARTIDO ISOLADO,MDB,AC,1981-06-25,3405502488,2,MASCULINO,8,SUPERIOR COMPLETO,3,CASADO(A),3,PARDA,278,VEREADOR,-1,#NULO,0.0,266330.0,0.0,50000.0,0.0,0.00,316330.00,12.664544,45
2,23/08/2026,19:30:42,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,AC,AC,ACRE,6,DEPUTADO FEDERAL,10002532418,1577,JOZINEY ALVES AMORIM,NEY AMORIM,#NULO,59549050297,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,15,MDB,MOVIMENTO DEMOCRÁTICO BRASILEIRO,-1,#NULO,#NULO,#NULO,10001800109,PARTIDO ISOLADO,MDB,AC,1977-01-24,2849542429,2,MASCULINO,8,SUPERIOR COMPLETO,3,CASADO(A),1,BRANCA,257,EMPRESÁRIO,-1,#NULO,3250000.0,0.0,0.0,0.0,0.0,0.00,3250000.00,14.994166,49
3,23/08/2026,19:30:42,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,AC,AC,ACRE,6,DEPUTADO FEDERAL,10002532419,1515,ANTÔNIA LUCILEIA CRUZ RAMOS CÂMARA,ANTÔNIA LÚCIA,#NULO,50791524272,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,15,MDB,MOVIMENTO DEMOCRÁTICO BRASILEIRO,-1,#NULO,#NULO,#NULO,10001800109,PARTIDO ISOLADO,MDB,AC,1970-07-17,16497022232,4,FEMININO,8,SUPERIOR COMPLETO,9,DIVORCIADO(A),3,PARDA,277,DEPUTADO,-1,#NULO,0.0,195000.0,0.0,0.0,0.0,3920000.00,4115000.00,15.230150,56
4,23/08/2026,19:30:42,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,AC,AC,ACRE,6,DEPUTADO FEDERAL,10002532420,1500,MINORU MARTINS KINPARA,MINORU KINPARA,#NULO,21722099291,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,15,MDB,MOVIMENTO DEMOCRÁTICO BRASILEIRO,-1,#NULO,#NULO,#NULO,10001800109,PARTIDO ISOLADO,MDB,GO,1968-12-02,1448152437,2,MASCULINO,8,SUPERIOR COMPLETO,3,CASADO(A),3,PARDA,142,PROFESSOR DE ENSINO SUPERIOR,-1,#NULO,300000.0,155000.0,0.0,0.0,0.0,1771244.57,2226244.57,14.615827,57


In [15]:
ANOS_ESTUDO_POR_GRAU = {
    'ANALFABETO': 0,
    'LÊ E ESCREVE': 1,
    'ENSINO FUNDAMENTAL INCOMPLETO': 4,
    'ENSINO FUNDAMENTAL COMPLETO': 8,
    'ENSINO MÉDIO INCOMPLETO': 9,
    'ENSINO MÉDIO COMPLETO': 11,
    'SUPERIOR INCOMPLETO': 13,
    'SUPERIOR COMPLETO': 16,
    # 'NÃO DIVULGÁVEL' fica de fora de propósito — mesma decisão da Versão 2 do notebook de clusterização
}

df['ANOS_ESTUDO'] = df['DS_GRAU_INSTRUCAO'].map(ANOS_ESTUDO_POR_GRAU)


In [16]:
colunas_driver = ['IDADE', 'ANOS_ESTUDO', 'Total_Bens_Log']

Q1, Q3 = df['Total_Bens_Log'].quantile([0.25, 0.75])
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

df_cluster = df[(df['Total_Bens_Log'] <= limite_superior) & (df['ANOS_ESTUDO'].notna())].copy()
print(f"Base para clusterização/regras: {df_cluster.shape[0]} candidatos "
      f"(descartados {df.shape[0] - df_cluster.shape[0]} por patrimônio extremo ou escolaridade não divulgada)")

scaler = MinMaxScaler()
X = scaler.fit_transform(df_cluster[colunas_driver])


Base para clusterização/regras: 925 candidatos (descartados 0 por patrimônio extremo ou escolaridade não divulgada)


In [17]:
def inercia(indices):
    if len(indices) < 2:
        return 0.0
    pontos = X[indices]
    centro = pontos.mean(axis=0)
    return float(((pontos - centro) ** 2).sum(axis=1).mean())


def bisecting_kmeans_ate_k(X, k_max, random_state=SEMENTE):
    """Mesma lógica do 02_clusterizacao.ipynb (Parte 2, critério de inércia média) — mas só
    devolve a partição final em k_max clusters, sem guardar o histórico de divisões (essa
    exploração já foi feita e documentada lá)."""
    clusters = {0: np.arange(X.shape[0])}
    proximo_id = 1
    for _ in range(1, k_max):
        id_escolhido = max(clusters, key=lambda cid: inercia(clusters[cid]))
        indices_pai = clusters[id_escolhido]
        if len(indices_pai) < 2:
            break
        km2 = KMeans(n_clusters=2, random_state=random_state, n_init=10)
        labels2 = km2.fit_predict(X[indices_pai])
        id_a, id_b = proximo_id, proximo_id + 1
        proximo_id += 2
        del clusters[id_escolhido]
        clusters[id_a] = indices_pai[labels2 == 0]
        clusters[id_b] = indices_pai[labels2 == 1]
    return clusters


k_bisecting = 3  # mesmo k escolhido no 02_clusterizacao.ipynb (Parte 2)

clusters_finais = bisecting_kmeans_ate_k(X, k_bisecting)
ids_por_tamanho = sorted(clusters_finais, key=lambda cid: len(clusters_finais[cid]), reverse=True)

rotulos = np.empty(X.shape[0], dtype=object)
for letra, cid in zip([chr(65 + i) for i in range(len(ids_por_tamanho))], ids_por_tamanho):
    rotulos[clusters_finais[cid]] = letra
df_cluster['cluster_bisecting'] = rotulos

# mesmos nomes definidos no 02_clusterizacao.ipynb, a partir da mesma ficha técnica
NOMES_CLUSTER_BISECTING = {
    'A': 'Com patrimônio',
    'B': 'Jovens sem bens',
    'C': 'Sem bens e alta escolaridade',
}
df_cluster['cluster'] = df_cluster['cluster_bisecting'].map(NOMES_CLUSTER_BISECTING)

df_cluster['cluster'].value_counts()


cluster
Com patrimônio                  622
Jovens sem bens                 158
Sem bens e alta escolaridade    145
Name: count, dtype: int64

## Montando a "cesta" de itens

Apriori não trabalha com números contínuos como o K-Means — ele precisa de **itens binários** (presente/ausente), do mesmo jeito que "pão" e "leite" são itens numa cesta de supermercado. Cada linha vira uma "cesta" (um candidato), e cada valor de cada variável categórica vira um "item" via one-hot encoding (`SG_PARTIDO=PT`, `DS_GENERO=FEMININO`, `cluster=Com patrimônio`, ...).

Variáveis escolhidas:
- `SG_PARTIDO`, `DS_GENERO`, `DS_GRAU_INSTRUCAO`, `DS_ESTADO_CIVIL`, `DS_COR_RACA`, `SG_UF_NASCIMENTO` — perfil do candidato.
- `cluster` — o nome do cluster da Fase 2 (Bisecting K-Means, k=3), recalculado acima. Incluir o cluster como item deixa a pergunta "o que caracteriza cada cluster?" ser respondida de novo do ponto de vista de regras — e permite achar combinações de 2+ variáveis que sozinhas não bastariam.
- `SG_UF` — aqui **entra**, diferente do que aconteceria com uma única UF: como `UF` é uma **lista de 7 estados** (Região Norte), a base tem candidatos de AC, AP, AM, PA, RO, RR e TO, então essa coluna discrimina (mesma lógica da visão Brasil do professor, `UF=None`) — só ficaria de fora se o recorte fosse uma UF só.
- `DS_SIT_TOT_TURNO` (eleito ou não) — só entra depois que a eleição de fato ocorrer (a coluna hoje vem toda com o mesmo valor, `#NULO`). A seção que usa isso fica pronta, mas roda condicionalmente lá na frente.

`DS_OCUPACAO` fica de fora por ora: são dezenas de categorias diferentes pra ~925 candidatos — a maioria das ocupações teria poucos candidatos, suporte baixo demais pra dizer qualquer coisa. Fica como exercício: agrupem em categorias maiores (ex.: "política", "direito", "saúde", "educação", "empresário"...) e testem.

In [18]:
colunas_apriori = ['SG_PARTIDO', 'DS_GENERO', 'DS_GRAU_INSTRUCAO', 'DS_ESTADO_CIVIL',
                   'DS_COR_RACA', 'SG_UF_NASCIMENTO', 'cluster']

# UF pode ser None (Brasil) ou uma lista de siglas (região, como aqui) -> mais de uma UF
# na base -> SG_UF discrimina. Só fica de fora quando UF é uma sigla única.
if UF is None or not isinstance(UF, str):
    colunas_apriori.append('SG_UF')

# a eleição de 2026 ainda não ocorreu -> DS_SIT_TOT_TURNO vem toda com o mesmo valor por enquanto
usar_resultado_eleicao = df_cluster['DS_SIT_TOT_TURNO'].nunique() > 1
if usar_resultado_eleicao:
    colunas_apriori.append('DS_SIT_TOT_TURNO')

cesta_base = df_cluster[['SQ_CANDIDATO'] + colunas_apriori].copy()
cesta = pd.get_dummies(cesta_base, columns=colunas_apriori, prefix_sep='=')
cesta = cesta.set_index('SQ_CANDIDATO')

print(f"{cesta.shape[0]} candidatos (cestas) x {cesta.shape[1]} itens possíveis (um por valor de cada variável)")
cesta_base.head()


925 candidatos (cestas) x 85 itens possíveis (um por valor de cada variável)


,SQ_CANDIDATO,SG_PARTIDO,DS_GENERO,DS_GRAU_INSTRUCAO,DS_ESTADO_CIVIL,DS_COR_RACA,SG_UF_NASCIMENTO,cluster,SG_UF
0,10002532416,MDB,FEMININO,SUPERIOR COMPLETO,CASADO(A),PARDA,AC,Com patrimônio,AC
1,10002532417,MDB,MASCULINO,SUPERIOR COMPLETO,CASADO(A),PARDA,AC,Com patrimônio,AC
2,10002532418,MDB,MASCULINO,SUPERIOR COMPLETO,CASADO(A),BRANCA,AC,Com patrimônio,AC
3,10002532419,MDB,FEMININO,SUPERIOR COMPLETO,DIVORCIADO(A),PARDA,AC,Com patrimônio,AC
4,10002532420,MDB,MASCULINO,SUPERIOR COMPLETO,CASADO(A),PARDA,GO,Com patrimônio,AC


## Itens frequentes: o que aparece sozinho, com que frequência

O Apriori funciona em duas etapas. Primeiro, encontra **itemsets frequentes** — combinações de 1, 2, 3... itens que aparecem juntas em pelo menos `min_suporte` das cestas. Só depois, na segunda etapa, ele vira **regras** (`A → B`). Começamos pelos itemsets de um item só — a frequência "crua" de cada valor, antes de cruzar qualquer coisa.


In [19]:
min_suporte = 0.02  # ~4 candidatos; ajuste se quiser regras mais raras (menor) ou mais comuns (maior)

itens_frequentes = apriori(cesta, min_support=min_suporte, use_colnames=True)
print(f"{len(itens_frequentes)} itemsets frequentes (min_suporte={min_suporte})")

individuais = itens_frequentes[itens_frequentes['itemsets'].apply(lambda x: len(x) == 1)].copy()
individuais['item'] = individuais['itemsets'].apply(lambda x: next(iter(x)))
individuais['qtd_candidatos'] = (individuais['support'] * len(cesta)).round().astype(int)
individuais.sort_values('support', ascending=False)[['item', 'support', 'qtd_candidatos']].head(15)


1369 itemsets frequentes (min_suporte=0.02)


,item,support,qtd_candidatos
44,cluster=Com patrimônio,0.672432,622
19,DS_GENERO=MASCULINO,0.605405,560
24,DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,0.588108,544
31,DS_COR_RACA=PARDA,0.528649,489
26,DS_ESTADO_CIVIL=CASADO(A),0.471351,436
18,DS_GENERO=FEMININO,0.394595,365
28,DS_ESTADO_CIVIL=SOLTEIRO(A),0.393514,364
29,DS_COR_RACA=BRANCA,0.326486,302
50,SG_UF=PA,0.283243,262
38,SG_UF_NASCIMENTO=PA,0.280000,259


### Suporte, confiança e lift — o que cada métrica mede

Regras de associação têm o formato **"se A, então B"** (A = antecedente, B = consequente), e três números resumem o quão forte é essa relação:

- **Suporte** — fração de candidatos em que A e B aparecem juntos: `suporte(A→B) = P(A e B)`. Mede o quão comum é a combinação; suporte baixo (poucos candidatos) é terreno fértil pra coincidência.
- **Confiança** — entre os candidatos que têm A, qual fração também tem B: `confiança(A→B) = P(B | A) = suporte(A e B) / suporte(A)`. É a "taxa de acerto" da regra.
- **Lift** — o quanto a confiança da regra é maior (ou menor) do que se A e B fossem independentes: `lift(A→B) = confiança(A→B) / suporte(B)`. `lift = 1` → A e B são independentes (a regra não diz nada); `lift > 1` → aparecem juntos mais do que o esperado ao acaso; `lift < 1` → aparecem juntos menos do que o esperado.

**Um cuidado importante nesta base:** ela tem 925 candidatos espalhados por 7 UFs da Região Norte e 28 partidos — em média, uns 132 candidatos por UF e uns 33 por partido. É uma base bem maior que a amostra nacional de Governador usada pelo professor (~200 candidatos), então o `min_suporte` de 2% já impõe um piso de ~18 candidatos por trás de qualquer item ou regra — bem mais robusto do que os ~4 candidatos que esse mesmo piso representava lá. Ainda assim, alguns cruzamentos ficam pequenos (a menor UF da região, o Amapá, tem só 89 candidatos), e a coluna `qtd_candidatos` continua a melhor forma de checar se um lift alto tem gente de verdade por trás.


In [20]:
regras = association_rules(itens_frequentes, metric='lift', min_threshold=1)
regras['qtd_candidatos'] = (regras['support'] * len(cesta)).round().astype(int)
regras = regras.sort_values('confidence', ascending=False)


def formatar_regras(df_regras):
    """Troca antecedents/consequents (frozenset) por texto legível, só pra exibição —
    os dados originais (usados pra filtrar/ordenar) continuam intactos."""
    df_fmt = df_regras.copy()
    df_fmt['antecedents'] = df_fmt['antecedents'].apply(lambda x: ', '.join(sorted(str(i) for i in x)))
    df_fmt['consequents'] = df_fmt['consequents'].apply(lambda x: ', '.join(sorted(str(i) for i in x)))
    return df_fmt


print(f"{len(regras)} regras (lift >= 1)")
formatar_regras(regras[['antecedents', 'consequents', 'qtd_candidatos', 'support', 'confidence', 'lift']].head(10))


9092 regras (lift >= 1)


,antecedents,consequents,qtd_candidatos,support,confidence,lift
1457,"DS_GENERO=MASCULINO, SG_UF_NASCIMENTO=AP",SG_UF=AP,32,0.034595,1.0,10.393258
6303,"DS_COR_RACA=PARDA, SG_UF_NASCIMENTO=TO, cluster=Com patrimônio",SG_UF=TO,24,0.025946,1.0,9.438776
3712,"DS_GENERO=MASCULINO, DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO, SG_UF=TO",cluster=Com patrimônio,20,0.021622,1.0,1.487138
5638,"DS_COR_RACA=PARDA, DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_UF_NASCIMENTO=AP",SG_UF=AP,27,0.029189,1.0,10.393258
7893,"DS_COR_RACA=BRANCA, DS_ESTADO_CIVIL=CASADO(A), DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_UF=RO",cluster=Com patrimônio,22,0.023784,1.0,1.487138
4339,"DS_ESTADO_CIVIL=CASADO(A), DS_GENERO=MASCULINO, SG_UF_NASCIMENTO=AP",SG_UF=AP,19,0.020541,1.0,10.393258
5231,"DS_ESTADO_CIVIL=CASADO(A), DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_UF_NASCIMENTO=AP",SG_UF=AP,25,0.027027,1.0,10.393258
6244,"DS_COR_RACA=PARDA, SG_UF_NASCIMENTO=AP, cluster=Com patrimônio",SG_UF=AP,22,0.023784,1.0,10.393258
1886,"DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_UF_NASCIMENTO=AP",SG_UF=AP,40,0.043243,1.0,10.393258
4306,"DS_ESTADO_CIVIL=CASADO(A), DS_GENERO=MASCULINO, SG_UF_NASCIMENTO=AC",SG_UF=AC,20,0.021622,1.0,9.068627


### Cuidado: suporte pequeno infla o lift

Vamos ver isso na prática: as regras de maior lift da base inteira, sem nenhum filtro de suporte, comparadas às regras de maior lift **exigindo pelo menos 10 candidatos** por trás delas.


In [21]:
regras_1_consequente = regras[regras['consequents'].apply(lambda x: len(x) == 1)].copy()
colunas_exibir = ['antecedents', 'consequents', 'qtd_candidatos', 'confidence', 'lift']

print("--- Maior lift, sem filtro de suporte ---")
display(formatar_regras(regras_1_consequente.sort_values('lift', ascending=False)[colunas_exibir].head(5)))

print("--- Maior lift, exigindo qtd_candidatos >= 10 ---")
display(formatar_regras(regras_1_consequente[regras_1_consequente['qtd_candidatos'] >= 10]
        .sort_values('lift', ascending=False)[colunas_exibir].head(5)))


--- Maior lift, sem filtro de suporte ---


,antecedents,consequents,qtd_candidatos,confidence,lift
5636,"DS_COR_RACA=PARDA, DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_UF=AP",SG_UF_NASCIMENTO=AP,27,0.750000,12.613636
8315,"DS_COR_RACA=PARDA, DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_UF=AP, cluster=Com patrimônio",SG_UF_NASCIMENTO=AP,19,0.730769,12.290210
5228,"DS_ESTADO_CIVIL=CASADO(A), DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_UF=AP",SG_UF_NASCIMENTO=AP,25,0.714286,12.012987
2404,"DS_COR_RACA=PARDA, SG_UF=AP",SG_UF_NASCIMENTO=AP,35,0.700000,11.772727
4337,"DS_ESTADO_CIVIL=CASADO(A), DS_GENERO=MASCULINO, SG_UF=AP",SG_UF_NASCIMENTO=AP,19,0.678571,11.412338


--- Maior lift, exigindo qtd_candidatos >= 10 ---


,antecedents,consequents,qtd_candidatos,confidence,lift
5636,"DS_COR_RACA=PARDA, DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_UF=AP",SG_UF_NASCIMENTO=AP,27,0.750000,12.613636
8315,"DS_COR_RACA=PARDA, DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_UF=AP, cluster=Com patrimônio",SG_UF_NASCIMENTO=AP,19,0.730769,12.290210
5228,"DS_ESTADO_CIVIL=CASADO(A), DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_UF=AP",SG_UF_NASCIMENTO=AP,25,0.714286,12.012987
2404,"DS_COR_RACA=PARDA, SG_UF=AP",SG_UF_NASCIMENTO=AP,35,0.700000,11.772727
4337,"DS_ESTADO_CIVIL=CASADO(A), DS_GENERO=MASCULINO, SG_UF=AP",SG_UF_NASCIMENTO=AP,19,0.678571,11.412338


Nos dois grupos, o topo da lista é dominado pelo mesmo padrão: `SG_UF=AP → SG_UF_NASCIMENTO=AP`, com lift acima de 11 (e até 12,6 quando combinado com escolaridade e cor/raça). Diferente do exemplo do professor, aqui isso não é só um artefato de amostra pequena — é **estrutural**: candidato a Deputado Federal concorre no estado onde tem domicílio eleitoral, que normalmente é também onde nasceu, então `SG_UF` e `SG_UF_NASCIMENTO` carregam quase a mesma informação. O efeito fica ainda mais forte nos estados menores da região (Amapá, Roraima, Acre): com menos candidatos "de fora", a correspondência UF↔naturalidade beira o determinístico (confiança perto de 1,0 em vários casos). E, como já era esperado pelo cuidado da célula anterior, aqui até a tabela "sem filtro" já exige ~18+ candidatos por trás de cada regra — então a diferença entre as duas tabelas é pequena, porque o piso do `min_suporte` já fazia boa parte desse trabalho sozinho.

### Filtrando pra regras que valem a pena olhar

In [22]:
suporte_pratico = 0.03    # ~6 candidatos
confianca_pratica = 0.5

regras_simples = regras[
    regras['antecedents'].apply(lambda x: len(x) <= 2) & regras['consequents'].apply(lambda x: len(x) == 1)
]
regras_confiaveis = regras_simples[
    (regras_simples['support'] >= suporte_pratico) & (regras_simples['confidence'] >= confianca_pratica)
]

print(f"{len(regras_confiaveis)} regras com suporte >= {suporte_pratico} e confiança >= {confianca_pratica}")
formatar_regras(regras_confiaveis.sort_values('confidence', ascending=False)[colunas_exibir].head(15))


498 regras com suporte >= 0.03 e confiança >= 0.5


,antecedents,consequents,qtd_candidatos,confidence,lift
1457,"DS_GENERO=MASCULINO, SG_UF_NASCIMENTO=AP",SG_UF=AP,32,1.000000,10.393258
1886,"DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_UF_NASCIMENTO=AP",SG_UF=AP,40,1.000000,10.393258
2106,"DS_ESTADO_CIVIL=CASADO(A), SG_UF_NASCIMENTO=AP",SG_UF=AP,30,1.000000,10.393258
341,SG_UF_NASCIMENTO=AP,SG_UF=AP,55,1.000000,10.393258
2509,"SG_UF_NASCIMENTO=AP, cluster=Com patrimônio",SG_UF=AP,30,1.000000,10.393258
2406,"DS_COR_RACA=PARDA, SG_UF_NASCIMENTO=AP",SG_UF=AP,35,1.000000,10.393258
356,SG_UF_NASCIMENTO=RR,SG_UF=RR,45,0.978261,8.618012
1449,"DS_GENERO=MASCULINO, SG_UF_NASCIMENTO=AC",SG_UF=AC,42,0.976744,8.857729
2259,"DS_ESTADO_CIVIL=SOLTEIRO(A), SG_UF_NASCIMENTO=AC",SG_UF=AC,37,0.973684,8.829979
2089,"DS_ESTADO_CIVIL=CASADO(A), SG_UF_NASCIMENTO=AC",SG_UF=AC,35,0.972222,8.816721


## O que prediz o cluster de um candidato?

Filtramos as regras cujo **consequente é um único item de cluster** — ou seja, "se o candidato tem tais características, ele cai no cluster X". É a mesma pergunta que a Fase 2 respondeu via ficha técnica, agora vista pelo ângulo de combinações de atributos categóricos.

Diferente do exemplo do professor (Governador Brasil, onde os clusters ficaram bem desbalanceados: 78% / 14% / 8%), aqui os três grupos são **mais equilibrados** — Com patrimônio (67%), Jovens sem bens (17%) e Sem bens e alta escolaridade (16%). Isso significa que nenhum dos três é tão raro a ponto de não sustentar regra nenhuma, mas o cluster majoritário (Com patrimônio) ainda tende a ter lift mais baixo (perto de 1,3–1,4) simplesmente por ser mais fácil de acertar por já ser a maioria. Por isso seguimos pegando as regras de **maior lift dentro de cada cluster**, separadamente — assim os três aparecem, e dá pra comparar a força relativa de cada um.

In [23]:
def regras_para_consequente(regras_completas, prefixo, n_por_valor=6, min_suporte=min_suporte, max_antecedentes=2):
    """
    Entre as regras cujo único item consequente comece com `prefixo` (ex.: 'cluster='),
    devolve as `n_por_valor` de MAIOR LIFT pra CADA valor distinto do consequente — sem piso
    de confiança fixo, porque um valor raro (poucos candidatos) não sustenta regras de
    confiança alta mesmo quando o lift é forte; ordenar só por confiança deixaria os valores
    mais comuns dominando a tabela inteira (ver célula anterior).
    """
    candidatas = regras_completas[
        regras_completas['consequents'].apply(lambda x: len(x) == 1 and next(iter(x)).startswith(prefixo))
        & regras_completas['antecedents'].apply(lambda x: len(x) <= max_antecedentes)
        & (regras_completas['support'] >= min_suporte)
    ].copy()
    candidatas['alvo'] = candidatas['consequents'].apply(lambda x: next(iter(x)))
    resultado = (candidatas.sort_values('lift', ascending=False)
                            .groupby('alvo', group_keys=False)
                            .head(n_por_valor)
                            .sort_values(['alvo', 'lift'], ascending=[True, False]))
    return resultado.drop(columns='alvo')


regras_cluster = regras_para_consequente(regras, 'cluster=')
print(f"{len(regras_cluster)} regras (até 6 por cluster, ordenadas por lift dentro de cada um)")
formatar_regras(regras_cluster[colunas_exibir])


18 regras (até 6 por cluster, ordenadas por lift dentro de cada um)


,antecedents,consequents,qtd_candidatos,confidence,lift
564,"DS_COR_RACA=BRANCA, SG_PARTIDO=PL",cluster=Com patrimônio,29,0.966667,1.437567
443,"DS_COR_RACA=PARDA, SG_PARTIDO=MDB",cluster=Com patrimônio,21,0.954545,1.419541
343,SG_UF_NASCIMENTO=GO,cluster=Com patrimônio,19,0.950000,1.412781
709,"DS_GENERO=MASCULINO, SG_PARTIDO=REPUBLICANOS",cluster=Com patrimônio,36,0.947368,1.408868
432,"DS_ESTADO_CIVIL=CASADO(A), SG_PARTIDO=MDB",cluster=Com patrimônio,31,0.939394,1.397009
752,"DS_COR_RACA=PARDA, SG_PARTIDO=REPUBLICANOS",cluster=Com patrimônio,27,0.931034,1.384577
1622,"DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO, SG_UF_NASCIMENTO=PA",cluster=Jovens sem bens,32,0.551724,3.230031
1557,"DS_ESTADO_CIVIL=SOLTEIRO(A), DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO",cluster=Jovens sem bens,54,0.540000,3.161392
33,SG_PARTIDO=MISSÃO,cluster=Jovens sem bens,19,0.527778,3.089838
1643,"DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO, SG_UF=PA",cluster=Jovens sem bens,31,0.525424,3.076057


### Visualizando como grafo

Uma tabela com dezenas de regras é difícil de escanear — um grafo interativo ajuda a ver de relance quais itens aparecem mais como "explicação" (antecedentes) e quais aparecem mais como "resultado" (consequentes). Cada item vira um nó (`SG_PARTIDO=PCO`, `cluster=D`, ...); cada regra vira uma seta do antecedente pro consequente, com espessura proporcional à confiança. Os nós que começam com `cluster=` ficam destacados em vermelho e maiores, pra serem fáceis de achar no meio dos outros itens. Passe o mouse sobre uma seta pra ver confiança, lift e quantos candidatos sustentam aquela regra — e arraste os nós se a rede ficar embaralhada.


In [24]:
def grafo_regras_html(regras_plot, arquivo_html, titulo, prefixo_destaque='cluster='):
    """
    Desenha as regras como uma rede interativa (pyvis) e grava em `arquivo_html`. Cada REGRA
    vira um nó de antecedente e um nó de consequente (quando o antecedente tem mais de um
    item, eles aparecem juntos no mesmo nó, separados por "+") ligados por UMA única aresta
    — nunca uma aresta por item isolado, pra não sugerir que um item sozinho sustenta a
    confiança/lift de uma regra que na verdade depende da combinação inteira (era isso que
    causava setas duplicadas entre o mesmo par de nós). Nós que contêm algum item começando
    com `prefixo_destaque` (por padrão, os clusters da Fase 2) ficam destacados em vermelho e
    maiores. Espessura da aresta = confiança da regra; passe o mouse pra ver confiança, lift
    e quantos candidatos a sustentam. Os controles de física (embaixo do grafo) deixam
    ajustar a repulsão/gravidade entre os nós ao vivo.

    Retorna um IFrame carregando o arquivo — se aparecer em branco/preto aqui embaixo (alguns
    ambientes bloqueiam o JavaScript de saídas de célula em notebooks "não confiáveis"), abra
    `arquivo_html` direto no navegador; o grafo é o mesmo.
    """
    print(titulo)
    net = Network(height='600px', width='100%', bgcolor='#222222', font_color='white',
                  notebook=True, directed=True, cdn_resources='in_line')

    nos_adicionados = set()
    for _, row in regras_plot.iterrows():
        antecedente = ' + '.join(sorted(str(item) for item in row['antecedents']))
        consequente = ' + '.join(sorted(str(item) for item in row['consequents']))

        for label, itemset in [(antecedente, row['antecedents']), (consequente, row['consequents'])]:
            if label in nos_adicionados:
                continue
            destaque = any(str(item).startswith(prefixo_destaque) for item in itemset)
            net.add_node(
                label, label, title=label,
                color='#e41a1c' if destaque else '#1f78b4',
                size=30 if destaque else 12,
                font={'size': 22 if destaque else 14},
            )
            nos_adicionados.add(label)

        titulo_aresta = (f"Confiança: {row['confidence']:.2f} | Lift: {row['lift']:.2f} | "
                          f"{row['qtd_candidatos']} candidatos")
        net.add_edge(antecedente, consequente, value=row['confidence'], title=titulo_aresta)

    net.show_buttons(filter_=['physics'])
    # net.save_graph grava com a codificação padrão do SO — no Windows isso quebra em nomes
    # com acento (ex.: "patrimônio"). Geramos o HTML e gravamos nós mesmos, sempre em UTF-8.
    html = net.generate_html(name=arquivo_html, notebook=True)
    with open(arquivo_html, 'w', encoding='utf-8') as f:
        f.write(html)
    print(f"Grafo salvo em: {arquivo_html} — se não aparecer abaixo, abra esse arquivo no navegador.")
    return IFrame(arquivo_html, width='100%', height=750)


grafo_regras_html(regras_cluster.head(20), 'grafo_regras_cluster.html', 'O que prediz o cluster de um candidato?')


O que prediz o cluster de um candidato?
Grafo salvo em: grafo_regras_cluster.html — se não aparecer abaixo, abra esse arquivo no navegador.


## O que caracteriza quem já é de um cluster?

A seção anterior fixou o **consequente** (`cluster=X`) e perguntou o que o prediz. Agora invertemos: fixamos o **antecedente** — "dado que o candidato é do cluster X, ..." — e olhamos pra que outros itens aparecem como consequente com lift alto. É uma pergunta diferente, não a mesma coisa ao contrário: uma regra `A → B` de lift alto não garante que `B → A` também tenha lift alto (confiança e suporte de cada lado são diferentes), então vale olhar as duas direções separadamente.


In [25]:
def regras_para_antecedente(regras_completas, item_fixo, n=8, min_suporte=min_suporte, max_consequentes=1):
    """
    Entre as regras cujo antecedente é EXATAMENTE {item_fixo} (um único item), devolve as
    `n` de maior lift. Direção oposta de `regras_para_consequente`: lá fixamos o "efeito"
    (o consequente) e procurávamos o que o prediz; aqui fixamos a "causa" (o antecedente) e
    olhamos o que ela prediz.
    """
    candidatas = regras_completas[
        regras_completas['antecedents'].apply(lambda x: len(x) == 1 and next(iter(x)) == item_fixo)
        & regras_completas['consequents'].apply(lambda x: len(x) <= max_consequentes)
        & (regras_completas['support'] >= min_suporte)
    ]
    return candidatas.sort_values('lift', ascending=False).head(n)


regras_por_cluster_fixo = {
    nome: regras_para_antecedente(regras, nome)
    for nome in ('cluster=Com patrimônio', 'cluster=Jovens sem bens', 'cluster=Sem bens e alta escolaridade')
}

for item_fixo, subset in regras_por_cluster_fixo.items():
    print(f"--- {item_fixo} ({len(subset)} regras) ---")
    display(formatar_regras(subset[colunas_exibir]))


--- cluster=Com patrimônio (8 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift
342,cluster=Com patrimônio,SG_UF_NASCIMENTO=GO,19,0.030547,1.412781
74,cluster=Com patrimônio,SG_PARTIDO=PP,19,0.030547,1.345506
24,cluster=Com patrimônio,SG_PARTIDO=MDB,50,0.080386,1.327802
95,cluster=Com patrimônio,SG_PARTIDO=PSDB,21,0.033762,1.301246
117,cluster=Com patrimônio,SG_PARTIDO=REPUBLICANOS,54,0.086817,1.295249
350,cluster=Com patrimônio,SG_UF_NASCIMENTO=PR,20,0.032154,1.293164
358,cluster=Com patrimônio,SG_UF_NASCIMENTO=SP,20,0.032154,1.293164
92,cluster=Com patrimônio,SG_PARTIDO=PSD,39,0.062701,1.288853


--- cluster=Jovens sem bens (8 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift
32,cluster=Jovens sem bens,SG_PARTIDO=MISSÃO,19,0.120253,3.089838
204,cluster=Jovens sem bens,DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO,84,0.531646,2.434516
101,cluster=Jovens sem bens,SG_PARTIDO=PSOL,19,0.120253,2.270085
238,cluster=Jovens sem bens,DS_GRAU_INSTRUCAO=SUPERIOR INCOMPLETO,30,0.189873,1.909053
279,cluster=Jovens sem bens,DS_ESTADO_CIVIL=SOLTEIRO(A),104,0.658228,1.672694
326,cluster=Jovens sem bens,DS_COR_RACA=PRETA,27,0.170886,1.534657
375,cluster=Jovens sem bens,SG_UF=PA,57,0.360759,1.273674
344,cluster=Jovens sem bens,SG_UF_NASCIMENTO=PA,56,0.354430,1.265823


--- cluster=Sem bens e alta escolaridade (8 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift
43,cluster=Sem bens e alta escolaridade,SG_PARTIDO=NOVO,22,0.151724,2.378726
381,cluster=Sem bens e alta escolaridade,SG_UF=AP,22,0.151724,1.576908
227,cluster=Sem bens e alta escolaridade,DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,132,0.910345,1.547921
383,cluster=Sem bens e alta escolaridade,SG_UF=PA,54,0.372414,1.314820
157,cluster=Sem bens e alta escolaridade,DS_GENERO=FEMININO,75,0.517241,1.310817
347,cluster=Sem bens e alta escolaridade,SG_UF_NASCIMENTO=PA,53,0.365517,1.305419
384,cluster=Sem bens e alta escolaridade,SG_UF=RR,19,0.131034,1.154351
379,cluster=Sem bens e alta escolaridade,SG_UF=AM,23,0.158621,1.070979


### Visualizando como grafo (antecedente fixo)

Mesmo grafo interativo de antes, só que agora as três "causas" (os clusters) são o ponto de partida fixo, e os itens que aparecem como consequência é que variam.


In [26]:
regras_antecedente_fixo = pd.concat(regras_por_cluster_fixo.values())
grafo_regras_html(regras_antecedente_fixo, 'grafo_regras_antecedente_fixo.html', 'O que caracteriza quem já é de um cluster?')


O que caracteriza quem já é de um cluster?
Grafo salvo em: grafo_regras_antecedente_fixo.html — se não aparecer abaixo, abra esse arquivo no navegador.


## E quando a eleição já tiver ocorrido?

Esta seção só roda depois que a Fase 0 for rodada de novo com o resultado oficial (`DS_SIT_TOT_TURNO` deixa de vir toda com o mesmo valor `#NULO`) — por enquanto, ela se anuncia e não faz nada. Repare que não precisamos escrever nenhum código novo: como `DS_SIT_TOT_TURNO` já entrou na cesta lá na preparação (quando aplicável), a mesma `regras_para_consequente` e o mesmo `grafo_regras_html` servem pra essa pergunta também.


In [27]:
if usar_resultado_eleicao:
    regras_eleicao = regras_para_consequente(regras, 'DS_SIT_TOT_TURNO=')
    print(f"{len(regras_eleicao)} regras apontando para o resultado da eleição")
    display(formatar_regras(regras_eleicao[colunas_exibir].head(15)))
    grafo_regras_html(regras_eleicao.head(20), 'grafo_regras_eleicao.html', 'O que prediz ser eleito?', prefixo_destaque='DS_SIT_TOT_TURNO=')
else:
    print("DS_SIT_TOT_TURNO ainda não tem informação real (a eleição de 2026 não ocorreu) — "
          "essa seção fica pronta pra usar assim que a Fase 0 for rodada de novo com o resultado oficial. "
          "Por enquanto, pulamos.")


DS_SIT_TOT_TURNO ainda não tem informação real (a eleição de 2026 não ocorreu) — essa seção fica pronta pra usar assim que a Fase 0 for rodada de novo com o resultado oficial. Por enquanto, pulamos.


---
## Fechamento e próximos passos

Resumindo o roteiro: transformamos cada candidato a Deputado Federal na Região Norte numa "cesta" de atributos categóricos (partido, gênero, escolaridade, cor/raça, UF de nascimento, UF de candidatura, e o cluster da Fase 2), minerar itemsets frequentes com o Apriori, geramos regras `A → B` com suporte/confiança/lift, e comparamos com o exemplo do professor (Governador, Brasil):

- O achado mais forte aqui é **estrutural**, não uma coincidência de amostra pequena: `SG_UF ≈ SG_UF_NASCIMENTO`, porque candidato a Deputado Federal concorre onde tem domicílio eleitoral — geralmente onde nasceu.
- Os clusters da Fase 2 ficaram bem mais equilibrados (67% / 17% / 16%) do que os do exemplo do professor (78% / 14% / 8%), então os três geraram regras razoavelmente interessantes.
- Padrões partidários específicos aparecem nos dois sentidos: `NOVO` puxa pra "Sem bens e alta escolaridade", `MISSÃO` puxa pra "Jovens sem bens"; e quem tem patrimônio (o grupo majoritário) tende a ter nascido fora da própria região (GO, PR, SP aparecem como UF de nascimento com lift acima de 1,29).

**Exercícios sugeridos:**
- Agrupar `DS_OCUPACAO` em categorias maiores e incluir na cesta.
- Variar `min_suporte` (célula da seção "Itens frequentes") e ver como o número de regras muda.
- Comparar com outra UF ou região, refazendo a Fase 0 — só muda a célula de configuração, do mesmo jeito que nas fases anteriores.
- Depois que a eleição ocorrer: rodar a Fase 0 de novo, e conferir se a seção "E quando a eleição já tiver ocorrido?" acima encontra regras fortes ligadas a `DS_SIT_TOT_TURNO=ELEITO`.

Próxima etapa do pipeline: `04_deteccao_anomalias.ipynb`.